In [ ]:
from pathlib import Path

import pandas as pd

from config.config import REGISTRIES_DIR, SUBREGISTRIES_DIR, VersionObject

DOCUMENT_REGISTRY: Path =  REGISTRIES_DIR / "document_registry.csv"
BASE_REGISTRY: Path = REGISTRIES_DIR / "base_output_registry.csv"
RAW_REGISTRY: Path = SUBREGISTRIES_DIR / "raw_file_output_registry.csv"
EXTRACTION_REGISTRY: Path = SUBREGISTRIES_DIR / "extraction_output_registry.csv"
CLASSIFICATION_REGISTRY: Path = SUBREGISTRIES_DIR / "DAS_classification_output_registry.csv"

version_object = VersionObject(pipeline_version="v1.0.0", software_version="v1.0.4")
df_document_registry = pd.read_csv(DOCUMENT_REGISTRY)
df_base_registry = pd.read_csv(BASE_REGISTRY)
df_raw_registry = pd.read_csv(RAW_REGISTRY)
df_extraction_registry = pd.read_csv(EXTRACTION_REGISTRY)
df_DAS_classification_registry = pd.read_csv(CLASSIFICATION_REGISTRY)

In [ ]:
merged_df_classification = df_DAS_classification_registry.merge(df_base_registry[["output_sha", "doc_doi"]], on="output_sha", how="left")

In [ ]:
duplicated = df_base_registry.drop(columns=["creation_date"]).duplicated().sum()

In [ ]:
duplicated["output_type"].value_counts()

In [ ]:
cols = df_base_registry.columns.drop("creation_date")

dup_mask = df_base_registry.duplicated(subset=cols, keep=False)

df_base_registry[dup_mask]["output_type"].value_counts()

In [ ]:
df_DAS_classification_registry.duplicated().sum()

In [ ]:
# Check the columns in merged_df_classification
print("Columns:", merged_df_classification.columns.tolist())

# Check what values exist in key columns
print("\nUnique section_type values:", merged_df_classification["section_type"].unique() if "section_type" in merged_df_classification.columns else "Column not found")
print("Unique stage values:", merged_df_classification["stage"].unique() if "stage" in merged_df_classification.columns else "Column not found")
print("Unique software_version values:", merged_df_classification["software_version"].unique() if "software_version" in merged_df_classification.columns else "Column not found")
print("\nversion_object.software_version =", version_object.software_version)

# Check the filter step-by-step
condition1 = merged_df_classification["section_type"]=="DAS"
condition2 = merged_df_classification["stage"]=="cleaned"
condition3 = merged_df_classification["software_version"]==version_object.software_version

print(f"\nRows matching condition1 (section_type=='DAS'): {condition1.sum()}")
print(f"Rows matching condition2 (stage=='cleaned'): {condition2.sum()}")
print(f"Rows matching condition3 (software_version=='{version_object.software_version}'): {condition3.sum()}")
print(f"Rows matching ALL conditions: {(condition1 & condition2 & condition3).sum()}")

In [ ]:
df_DAS_classification_registry = pd.read_csv(CLASSIFICATION_REGISTRY)
df_classifications = df_DAS_classification_registry.loc[df_DAS_classification_registry["model"]=="claude-haiku-4-5-20251001", "label"]
df_classifications.duplicated().sum()

In [ ]:
merge_base_classification = df_DAS_classification_registry.merge(df_base_registry[["output_sha", "software_version"]], on="output_sha", how="left")
df_haiku_latest = merge_base_classification.loc[merge_base_classification["software_version"]=="v1.0.2"]
df_haiku_latest.value_counts(subset="label")

In [ ]:
df_haiku_latest.duplicated().sum()

In [ ]:
merge_base_classification.duplicated().sum()

In [ ]:
# Check for duplicate output_sha in the right dataframe
print(df_base_registry["output_sha"].duplicated().sum())

# Check total rows after merge vs expected
print(f"Rows after merge: {len(merge_base_classification)}")
print(f"Rows in haiku_latest: {len(df_haiku_latest)}")

# Check for NaN labels (the missing count)
print(f"NaN labels: {df_haiku_latest['label'].isna().sum()}")

In [ ]:
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt

counts = df_haiku_latest.value_counts(subset="label")
total = counts.sum()
pcts = (counts / total * 100).round(1)

labels = counts.index.tolist()
values = counts.values
colors = ['#888780', '#D85A30', '#1D9E75', '#7F77DD', '#FAC775', '#F09595', '#D4537E']

fig = plt.figure(figsize=(14, 5))
fig.suptitle(f"DAS classification results — v1.0.2  (n={total:,})", fontsize=13, fontweight='500', x=0.02, ha='left')
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.4)

# --- metric cards (text axes) ---
card_data = [
    ("total sections", f"{total:,}", "#333"),
    ("open access",    f"{pcts.get('open_access', 0):.1f}%", "#1D9E75"),
    ("no data",        f"{pcts.get('no_data', 0):.1f}%", "#888780"),
    ("restricted",     f"{pcts.get('restricted_access', 0):.1f}%", "#D85A30"),
]

ax_cards = fig.add_subplot(gs[0, 0])
ax_cards.axis('off')
for i, (label, val, color) in enumerate(card_data):
    y = 0.85 - i * 0.22
    ax_cards.text(0.05, y,      label, fontsize=9,  color='gray',  transform=ax_cards.transAxes)
    ax_cards.text(0.05, y-0.09, val,   fontsize=16, color=color, fontweight='bold', transform=ax_cards.transAxes)

# --- donut ---
ax_donut = fig.add_subplot(gs[0, 1])
wedges, _ = ax_donut.pie(values, colors=colors, startangle=90,
                          wedgeprops=dict(width=0.5, edgecolor='white', linewidth=1.5))
ax_donut.set_title("distribution", fontsize=10, color='gray', pad=10)

# --- horizontal bar ---
ax_bar = fig.add_subplot(gs[0, 2])
bars = ax_bar.barh(labels[::-1], pcts.values[::-1], color=colors[::-1], height=0.6)
ax_bar.set_xlabel("% of sections", fontsize=9, color='gray')
ax_bar.tick_params(axis='y', labelsize=9)
ax_bar.tick_params(axis='x', labelsize=9, colors='gray')
ax_bar.spines[['top', 'right', 'left']].set_visible(False)
ax_bar.xaxis.grid(True, linestyle='--', alpha=0.4)
ax_bar.set_axisbelow(True)

for bar, pct, cnt in zip(bars, pcts.values[::-1], values[::-1]):
    ax_bar.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f"{pct}%  ({cnt:,})", va='center', fontsize=8, color='gray')

ax_bar.set_xlim(0, pcts.max() + 15)

plt.tight_layout()
plt.savefig("das_classification_v102.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

has_data = df_haiku_latest[df_haiku_latest["label"] != "no_data"]
counts = has_data.value_counts(subset="label")
total = counts.sum()
pcts = (counts / total * 100).round(1)

labels = counts.index.tolist()
values = counts.values
colors = ['#D85A30', '#1D9E75', '#7F77DD', '#FAC775', '#F09595', '#D4537E']

fig, ax = plt.subplots(figsize=(8, 4))
fig.suptitle(
    f"DAS classification — studies with associated data  (n={total:,} / {len(df_haiku_latest):,} total)",
    fontsize=11, x=0.02, ha='left'
)

bars = ax.barh(labels[::-1], pcts.values[::-1], color=colors[::-1], height=0.55)

ax.set_xlabel("% of sections with data", fontsize=9, color='gray')
ax.tick_params(axis='y', labelsize=9)
ax.tick_params(axis='x', labelsize=9, colors='gray')
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.xaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)

for bar, pct, cnt in zip(bars, pcts.values[::-1], values[::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{pct}%  ({cnt:,})", va='center', fontsize=9, color='gray')

ax.set_xlim(0, pcts.max() + 18)

plt.tight_layout()
plt.savefig("das_classification_with_data.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

counts = df_document_registry["has_DAS"].value_counts()
total = counts.sum()
has = counts.get(True, 0)
has_not = counts.get(False, 0)

fig, ax = plt.subplots(figsize=(6, 3))
fig.suptitle(
    f"DAS presence across studies  (n={total:,})",
    fontsize=11, x=0.02, ha='left'
)

bars = ax.barh(
    ["no DAS", "has DAS"],
    [has_not / total * 100, has / total * 100],
    color=['#888780', '#1D9E75'], height=0.45
)

ax.set_xlabel("% of studies", fontsize=9, color='gray')
ax.tick_params(axis='y', labelsize=9)
ax.tick_params(axis='x', labelsize=9, colors='gray')
ax.spines[['top', 'right', 'left']].set_visible(False)
ax.xaxis.grid(True, linestyle='--', alpha=0.4)
ax.set_axisbelow(True)

for bar, cnt in zip(bars, [has_not, has]):
    pct = cnt / total * 100
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f"{pct:.1f}%  ({cnt:,})", va='center', fontsize=9, color='gray')

ax.set_xlim(0, 100 + 12)

plt.tight_layout()
plt.savefig("das_presence.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Check duplicates in each dataframe independently
print("Duplicates in df_classification_registry:", df_DAS_classification_registry["output_sha"].duplicated().sum())
print("Duplicates in df_base_registry:", df_base_registry["output_sha"].duplicated().sum())

# If df_base_registry is the culprit, see how many per sha
dup_counts = df_base_registry[df_base_registry["output_sha"].duplicated(keep=False)] \
    .groupby("output_sha") \
    .size() \
    .sort_values(ascending=False)
print(dup_counts.head(20))

# Also check if the duplicates differ in software_version 
# (i.e. same sha processed across multiple runs)
df_base_registry[df_base_registry["output_sha"].duplicated(keep=False)] \
    .groupby("output_sha")["software_version"] \
    .nunique() \
    .value_counts()

In [ ]:
dup_shas = df_base_registry[df_base_registry["output_sha"].duplicated(keep=False)]

# Case 1: same sha, different versions (reprocessing issue)
cross_version = dup_shas.groupby("output_sha").filter(
    lambda x: x["software_version"].nunique() > 1
)
print(cross_version.sort_values("output_sha").head(10))

# Case 2: same sha, same version (collision or double write)
same_version = dup_shas.groupby("output_sha").filter(
    lambda x: x["software_version"].nunique() == 1
)
print(same_version.sort_values("output_sha").head(10))

# For case 2, check if ALL columns are identical (pure double write)
# or if they differ (actual SHA collision)
same_version_dupes = same_version.drop_duplicates()
print(f"Rows remaining after full dedup: {len(same_version_dupes)}")
print(f"Original duplicate rows: {len(same_version)}")

In [ ]:
import matplotlib.pyplot as plt

df_das_context = (
    df_haiku_latest
    .merge(df_base_registry[["output_sha", "doc_doi"]], on="output_sha", how="left")
    .merge(df_document_registry[["doc_doi", "journal", "publication_year"]], on="doc_doi", how="left")
)
df_das_context["journal"] = df_das_context["journal"].fillna("Unknown journal")

label_order = [
    "restricted_access",
    "open_access",
    "unclear",
    "partial_access",
    "no_access",
    "incorrect",
]
label_colors = {
    "restricted_access": "#D85A30",
    "open_access": "#1D9E75",
    "unclear": "#7F77DD",
    "partial_access": "#FAC775",
    "no_access": "#F09595",
    "incorrect": "#D4537E",
}

df_das_with_data = df_das_context[df_das_context["label"] != "no_data"].copy()
journal_order = df_das_with_data["journal"].value_counts().index.tolist()

das_by_journal = (
    pd.crosstab(df_das_with_data["journal"], df_das_with_data["label"], normalize="index")
    .reindex(index=journal_order)
    .reindex(columns=label_order, fill_value=0)
    * 100
).round(1)
journal_totals = df_das_with_data["journal"].value_counts().reindex(journal_order)

fig, ax = plt.subplots(figsize=(10.5, 4.8))
left = pd.Series(0.0, index=das_by_journal.index)

for label in label_order:
    widths = das_by_journal[label]
    ax.barh(
        das_by_journal.index,
        widths,
        left=left,
        color=label_colors[label],
        edgecolor="white",
        height=0.65,
        label=label,
    )
    left = left + widths

ax.invert_yaxis()
ax.set_xlabel("% of DAS sections, excluding no_data", fontsize=9, color="gray")
ax.set_ylabel("")
ax.tick_params(axis="y", labelsize=9)
ax.tick_params(axis="x", labelsize=9, colors="gray")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.xaxis.grid(True, linestyle="--", alpha=0.35)
ax.set_axisbelow(True)

for journal, total in journal_totals.items():
    ax.text(101.5, journal, f"n={total:,}", va="center", fontsize=9, color="gray")

ax.set_xlim(0, 112)
ax.legend(title="DAS label", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
fig.suptitle(
    f"DAS classification by journal, excluding no_data (n={len(df_das_with_data):,})",
    fontsize=12,
    x=0.02,
    ha="left",
)

plt.tight_layout()
plt.savefig("das_classification_by_journal_excluding_no_data.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

if "df_das_context" not in globals():
    df_das_context = (
        df_haiku_latest
        .merge(df_base_registry[["output_sha", "doc_doi"]], on="output_sha", how="left")
        .merge(df_document_registry[["doc_doi", "journal", "publication_year"]], on="doc_doi", how="left")
    )
df_das_context["journal"] = df_das_context["journal"].fillna("Unknown journal")

if "label_order" not in globals() or "restricted_access" not in label_order:
    label_order = [
        "restricted_access",
        "open_access",
        "unclear",
        "partial_access",
        "no_access",
        "incorrect",
    ]

df_das_with_data = df_das_context[df_das_context["label"] != "no_data"].copy()
if "journal_order" not in globals():
    journal_order = df_das_with_data["journal"].value_counts().index.tolist()

das_counts_by_journal = (
    pd.crosstab(df_das_with_data["journal"], df_das_with_data["label"])
    .reindex(index=journal_order)
    .reindex(columns=label_order, fill_value=0)
)
journal_totals = das_counts_by_journal.sum(axis=1)

count_cmap = LinearSegmentedColormap.from_list(
    "das_counts",
    ["#f7f4ea", "#d7d1bc", "#9c9485", "#5c554c"],
)

fig_width = max(10, 1.4 * len(das_counts_by_journal.columns) + 3)
fig_height = max(4.8, 0.8 * len(das_counts_by_journal.index) + 1.8)
fig, ax = plt.subplots(figsize=(fig_width, fig_height))

im = ax.imshow(das_counts_by_journal.values, cmap=count_cmap, aspect="auto")

ax.set_xticks(range(len(das_counts_by_journal.columns)))
ax.set_xticklabels(das_counts_by_journal.columns, rotation=25, ha="right", fontsize=9)
ax.set_yticks(range(len(das_counts_by_journal.index)))
ax.set_yticklabels(das_counts_by_journal.index, fontsize=9)
ax.tick_params(top=False, bottom=True, labeltop=False, labelbottom=True)

max_value = das_counts_by_journal.values.max() if das_counts_by_journal.values.size else 0
for i, journal in enumerate(das_counts_by_journal.index):
    for j, label in enumerate(das_counts_by_journal.columns):
        value = int(das_counts_by_journal.iloc[i, j])
        text_color = "white" if max_value and value > max_value * 0.45 else "#2f2a24"
        ax.text(j, i, f"{value}", ha="center", va="center", color=text_color, fontsize=9, fontweight="bold")

for i, total in enumerate(journal_totals):
    ax.text(
        len(das_counts_by_journal.columns) - 0.05,
        i,
        f" total={int(total):,}",
        ha="left",
        va="center",
        fontsize=9,
        color="gray",
        clip_on=False,
    )

ax.set_xticks([x - 0.5 for x in range(1, len(das_counts_by_journal.columns))], minor=True)
ax.set_yticks([y - 0.5 for y in range(1, len(das_counts_by_journal.index))], minor=True)
ax.grid(which="minor", color="white", linestyle="-", linewidth=1.5)
ax.tick_params(which="minor", bottom=False, left=False)
for spine in ax.spines.values():
    spine.set_visible(False)

cbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.03)
cbar.ax.tick_params(labelsize=8, colors="gray")
cbar.outline.set_visible(False)
cbar.set_label("count", fontsize=9, color="gray")

fig.suptitle("DAS classification by journal - category counts excluding no_data", fontsize=12, x=0.02, ha="left")
plt.tight_layout()
plt.savefig("das_classification_by_journal_counts_excluding_no_data.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
import matplotlib.pyplot as plt

if "df_das_context" not in globals():
    df_das_context = (
        df_haiku_latest
        .merge(df_base_registry[["output_sha", "doc_doi"]], on="output_sha", how="left")
        .merge(df_document_registry[["doc_doi", "journal", "publication_year"]], on="doc_doi", how="left")
    )

year_labels = df_das_context["publication_year"].astype("Int64").astype(str).replace("<NA>", "Unknown year")
year_order = sorted([label for label in year_labels.unique() if label != "Unknown year"], key=int)
if (year_labels == "Unknown year").any():
    year_order.append("Unknown year")

das_by_year = (
    pd.crosstab(year_labels, df_das_context["label"], normalize="index")
    .reindex(index=year_order)
    .reindex(columns=label_order, fill_value=0)
    * 100
).round(1)
year_totals = year_labels.value_counts().reindex(year_order)

fig, ax = plt.subplots(figsize=(10, max(4.5, 0.6 * len(year_order) + 1.5)))
left = pd.Series(0.0, index=das_by_year.index)

for label in label_order:
    widths = das_by_year[label]
    ax.barh(
        das_by_year.index,
        widths,
        left=left,
        color=label_colors[label],
        edgecolor="white",
        height=0.65,
        label=label,
    )
    left = left + widths

ax.invert_yaxis()
ax.set_xlabel("% of DAS sections", fontsize=9, color="gray")
ax.set_ylabel("")
ax.tick_params(axis="y", labelsize=9)
ax.tick_params(axis="x", labelsize=9, colors="gray")
ax.spines[["top", "right", "left"]].set_visible(False)
ax.xaxis.grid(True, linestyle="--", alpha=0.35)
ax.set_axisbelow(True)

for year, total in year_totals.items():
    ax.text(101.5, year, f"n={total:,}", va="center", fontsize=9, color="gray")

ax.set_xlim(0, 110)
ax.legend(title="DAS label", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
fig.suptitle("DAS classification by publication year", fontsize=12, x=0.02, ha="left")

plt.tight_layout()
plt.savefig("das_classification_by_year.png", dpi=150, bbox_inches="tight")
plt.show()
